<a href="https://colab.research.google.com/github/swabhimansahu2004/Hybrid-TinyML-Environmental-Monitoring/blob/Swabhiman_Teacher_Model/Phase_5_Quantization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import numpy as np
import pandas as pd
import os

with tfmot.sparsity.keras.prune_scope():
  final_pruned_model = tf.keras.models.load_model('student_pruned.h5')
  print("Sparse Student Model loaded into Memory")

Sparse Student Model loaded into Memory


In [2]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# 1. Load your original dataset
df = pd.read_csv('smoke_detection_iot.csv')

# 2. Select the exact 12 features used in training
features = ['Humidity[%]', 'Pressure[hPa]', 'Raw H2', 'NC2.5', 'NC1.0',
            'PM2.5', 'eCO2[ppm]', 'PM1.0', 'NC0.5', 'Temperature[C]',
            'TVOC[ppb]', 'Raw Ethanol']

X = df[features]
y = df['Fire Alarm']

# 3. Split the data exactly as you did in Phase 1 & 2
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.125, random_state=42)

# 4. Scale the data (This is what the representative_data_gen needs!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print("✅ X_train_scaled is now defined and ready for the converter.")

✅ X_train_scaled is now defined and ready for the converter.


In [5]:
import tensorflow as tf
import os

# 1. Fixed Representative Data Generator
def representative_data_gen():
    # We use X_train_scaled because it is already a NumPy array
    # and has the correct normalization applied.
    for i in range(100):
        # Accessing row 'i', adding batch dimension, and ensuring float32
        sample = np.expand_dims(X_train_scaled[i], axis=0).astype(np.float32)
        yield [sample]

# 2. Initialize Converter with your Stripped Model
converter = tf.lite.TFLiteConverter.from_keras_model(final_pruned_model)

# 3. Apply Optimizations
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# 4. Enforce Integer-only operations (Critical for ESP32/Wokwi)
# This ensures the model uses 8-bit math instead of 32-bit floats
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

# 5. Convert and Save
try:
    tflite_model = converter.convert()
    with open("fire_model_optimized.tflite", "wb") as f:
        f.write(tflite_model)

    size_kb = os.path.getsize('fire_model_optimized.tflite') / 1024
    print(f"✅ FINAL SUCCESS!")
    print(f"Quantized TFLite Size: {size_kb:.2f} KB")
except Exception as e:
    print(f"❌ Conversion failed: {e}")

✅ FINAL SUCCESS!
Quantized TFLite Size: 3.69 KB


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [6]:
# Function to convert binary TFLite to a C++ hex array
def convert_to_c_array(contents, name):
    out = [f'unsigned char {name}[] = {{']
    l = [f'  0x{b:02x}' for b in contents]
    # Format 12 hex values per line for readability
    for i in range(0, len(l), 12):
        out.append(", ".join(l[i:i+12]) + ",")
    out.append(f'}};')
    out.append(f'unsigned int {name}_len = {len(contents)};')
    return "\n".join(out)

# Generate the content
model_h_content = convert_to_c_array(tflite_model, "fire_model_data")

# Save to model.h
with open("model.h", "w") as f:
    f.write(model_h_content)

print("🎉 'model.h' has been generated successfully!")
print("1. Look at the sidebar (Folder icon) in Colab.")
print("2. Download 'model.h'.")
print("3. This file contains your 3.69 KB optimized brain.")

🎉 'model.h' has been generated successfully!
1. Look at the sidebar (Folder icon) in Colab.
2. Download 'model.h'.
3. This file contains your 3.69 KB optimized brain.
